# Foundation Result Reproduction

This notebook repeats the historical three-fold foundation screen recorded in foundation-v1. It never promotes a model or changes the historical run.

## Before you start

1. Use a clean remote CUDA environment and run uv sync --locked.
2. Use a GPU with at least 10 GiB of free VRAM.
3. Run the cells in order. The final cell writes PASS or FAIL and raises an error if the result differs.

## Recorded reference GPU

The historical run used an NVIDIA GeForce RTX 3060 with 12 GB VRAM. PyTorch reported 11.63 GiB total and 11.52 GiB free before training; the preflight peak allocation was 8.59 GiB. Other compatible GPUs are allowed, but they must still reproduce the metrics exactly.

## Steps performed by this notebook

1. Set a new, empty output run name and load the foundation-v1 reference.
2. Check the lockfile, package versions, CUDA memory, data IDs, labels, folds, and saved CTR and TabM recipes.
3. Run the existing preflight and training pipeline with the recorded configuration.
4. Compare the selected candidate, artifacts, OOF rows, and every raw-OOF metric with exact equality.

The notebook records the GPU details in the reproduction evidence. A GPU different from the reference RTX 3060 does not relax the exact-match requirement.

In [ ]:
from dataclasses import replace
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold
import torch

from prs_its.foundation_training import (
    FoundationTrainingConfig,
    preflight_foundation_training,
    run_foundation_training,
)
from prs_its.metrics import evaluate_probabilities
from prs_its.modeling import ID_COL, TARGET, make_feature_spec, validate_train_test_schema
from prs_its.training import find_project_root, load_competition_data


In [ ]:
ROOT = find_project_root(Path.cwd())
REFERENCE_RUN_NAME = "foundation-v1"
REPRODUCTION_RUN_NAME = "foundation-repro-v1"
REFERENCE_ROOT = ROOT / "outputs" / "runs" / REFERENCE_RUN_NAME
REPRODUCTION_ROOT = ROOT / "outputs" / "runs" / REPRODUCTION_RUN_NAME
REFERENCE_METRICS = REFERENCE_ROOT / "metrics"
REFERENCE_OOF = REFERENCE_ROOT / "oof"

if REPRODUCTION_RUN_NAME == REFERENCE_RUN_NAME:
    raise ValueError("The reproduction run name must differ from foundation-v1.")
if REPRODUCTION_ROOT.exists() and any(REPRODUCTION_ROOT.iterdir()):
    raise FileExistsError(
        f"Reproduction output already contains artifacts: {REPRODUCTION_ROOT}. "
        "Choose a new REPRODUCTION_RUN_NAME for a clean replay."
    )

with (REFERENCE_METRICS / "foundation_final_config.json").open() as handle:
    REFERENCE_CONFIG = json.load(handle)
with (REFERENCE_METRICS / "foundation_promotion_decision.json").open() as handle:
    REFERENCE_DECISION = json.load(handle)

EXPECTED_CANDIDATE = REFERENCE_DECISION["selected_experiment"]
EXPECTED_FOUNDATION_WEIGHT = REFERENCE_DECISION["selected_foundation_weight"]
FOUNDATION_PARAMS = REFERENCE_CONFIG["foundation_params"]
CV_CONFIG = REFERENCE_CONFIG["cv"]

REPRODUCTION_CONFIG = FoundationTrainingConfig(
    project_root=ROOT,
    run_name=REPRODUCTION_RUN_NAME,
    incumbent_run_name=REFERENCE_CONFIG["ctr_source"]["run_name"],
    tabm_run_name=REFERENCE_CONFIG["tabm_source"]["run_name"],
    model=FOUNDATION_PARAMS["model"],
    task_type="GPU",
    devices="0",
    n_splits=CV_CONFIG["n_splits"],
    random_state=CV_CONFIG["random_state"],
    max_runtime_minutes=360.0,
    n_bootstrap=1000,
    show_progress=True,
    n_estimators=FOUNDATION_PARAMS["n_estimators"],
    prediction_chunk_size=FOUNDATION_PARAMS["prediction_chunk_size"],
    min_free_vram_gib=FOUNDATION_PARAMS["min_free_vram_gib"],
    tabicl_cache_mode=FOUNDATION_PARAMS["tabicl_cache_mode"],
    resume=False,
)

print(f"Reference run: {REFERENCE_ROOT}")
print(f"Reproduction run: {REPRODUCTION_ROOT}")
print(f"Expected candidate: {EXPECTED_CANDIDATE}")

## Step 2 — Check prerequisites

These checks fail before any new run artifacts are created. The data hashes are retained as evidence, while IDs, labels, fold assignments, feature schema, and source recipes are compared directly with the historical artifacts.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def installed_version(distribution: str) -> str:
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return "not-installed"


lock_check = subprocess.run(
    ["uv", "lock", "--check"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
if lock_check.returncode != 0:
    raise RuntimeError(lock_check.stderr.strip() or lock_check.stdout.strip())

expected_versions = REFERENCE_CONFIG["versions"]
actual_versions = {
    "python": platform.python_version(),
    "numpy": installed_version("numpy"),
    "pandas": installed_version("pandas"),
    "torch": installed_version("torch"),
    "tabicl": installed_version("tabicl"),
    "tabpfn": installed_version("tabpfn"),
    "scikit-learn": installed_version("scikit-learn"),
}
version_mismatches = {
    name: {"expected": expected_versions[name], "actual": actual_versions[name]}
    for name in expected_versions
    if actual_versions[name] != expected_versions[name]
}
if version_mismatches:
    raise RuntimeError(f"Recorded package versions do not match: {version_mismatches}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this reproduction run.")
device = torch.device("cuda")
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
required_bytes = int(FOUNDATION_PARAMS["min_free_vram_gib"] * 1024**3)
if free_bytes < required_bytes:
    raise RuntimeError(
        f"At least {FOUNDATION_PARAMS['min_free_vram_gib']:.1f} GiB free VRAM is required; "
        f"only {free_bytes / 1024**3:.2f} GiB is available."
    )

train, test = load_competition_data(ROOT)
features = validate_train_test_schema(train, test)
spec = make_feature_spec(train, test)
if features != REFERENCE_CONFIG["features"]:
    raise ValueError("The current feature schema does not match foundation-v1.")
if spec.categorical_features != REFERENCE_CONFIG["categorical_features"]:
    raise ValueError("The current categorical feature schema does not match foundation-v1.")
if set(spec.excluded_features) != {ID_COL} or set(REFERENCE_CONFIG["excluded_features"]) != {ID_COL, TARGET}:
    raise ValueError("The current excluded feature schema is incompatible with foundation-v1.")

reference_oof = pd.read_csv(REFERENCE_OOF / f"{EXPECTED_CANDIDATE}_oof.csv")
reference_test = pd.read_csv(REFERENCE_OOF / f"{EXPECTED_CANDIDATE}_test_raw.csv")
if not np.array_equal(reference_oof[ID_COL].to_numpy(), train[ID_COL].to_numpy()):
    raise ValueError("Training claim IDs do not match the historical OOF artifact.")
if not np.array_equal(reference_oof[TARGET].to_numpy(dtype=int), train[TARGET].to_numpy(dtype=int)):
    raise ValueError("Training labels do not match the historical OOF artifact.")
if not np.array_equal(reference_test[ID_COL].to_numpy(), test[ID_COL].to_numpy()):
    raise ValueError("Test claim IDs do not match the historical raw prediction artifact.")

cv = StratifiedKFold(
    n_splits=CV_CONFIG["n_splits"],
    shuffle=CV_CONFIG["shuffle"],
    random_state=CV_CONFIG["random_state"],
)
expected_folds = np.full(len(train), -1, dtype=int)
for fold, (_, valid_idx) in enumerate(cv.split(train, train[TARGET].astype(int))):
    expected_folds[valid_idx] = fold
if not np.array_equal(reference_oof["fold"].to_numpy(dtype=int), expected_folds):
    raise ValueError("The historical OOF folds do not match the recorded cross-validation configuration.")

with (ROOT / "outputs" / "runs" / REPRODUCTION_CONFIG.incumbent_run_name / "models" / "catboost_final_config.json").open() as handle:
    ctr_config = json.load(handle)
with (ROOT / "outputs" / "runs" / REPRODUCTION_CONFIG.tabm_run_name / "metrics" / "tabm_hpo_selection.json").open() as handle:
    tabm_selection = json.load(handle)
if ctr_config["experiment"] != REFERENCE_CONFIG["ctr_source"]["recipe"]:
    raise ValueError("The CTR source recipe does not match foundation-v1.")
if ctr_config["params"] != REFERENCE_CONFIG["ctr_source"]["params"]:
    raise ValueError("The CTR source parameters do not match foundation-v1.")
if tabm_selection.get("selected_candidate") != REFERENCE_CONFIG["tabm_source"]["selection"]:
    raise ValueError("The TabM source selection does not match foundation-v1.")

ENVIRONMENT_EVIDENCE = {
    "reference_run": REFERENCE_RUN_NAME,
    "reproduction_run": REPRODUCTION_RUN_NAME,
    "uv_lock_check": lock_check.stdout.strip() or "passed",
    "versions": actual_versions,
    "gpu": {
        "name": torch.cuda.get_device_name(device),
        "device_count": torch.cuda.device_count(),
        "torch_cuda": torch.version.cuda,
        "free_vram_gib": free_bytes / 1024**3,
        "total_vram_gib": total_bytes / 1024**3,
    },
    "data": {
        "train_rows": len(train),
        "test_rows": len(test),
        "train_sha256": sha256_file(ROOT / "data" / "train.csv"),
        "test_sha256": sha256_file(ROOT / "data" / "test.csv"),
    },
}

display(pd.DataFrame([ENVIRONMENT_EVIDENCE["gpu"]]))
display(pd.DataFrame([ENVIRONMENT_EVIDENCE["versions"]]))

## Step 3 — Run the historical configuration

The production preflight creates the fresh output directory and saves its result. Training then uses explicit resume mode so it consumes that exact preflight artifact rather than rerunning it.

In [ ]:
preflight = preflight_foundation_training(REPRODUCTION_CONFIG)
with (REPRODUCTION_ROOT / "metrics" / "reproduction_environment.json").open("w") as handle:
    json.dump(ENVIRONMENT_EVIDENCE, handle, indent=2, sort_keys=True)

training_result = run_foundation_training(replace(REPRODUCTION_CONFIG, resume=True))
display(pd.DataFrame([preflight]))
display(pd.DataFrame([{key: str(value) for key, value in training_result.items()}]))

## Step 4 — Verify the exact raw-OOF result

The comparison covers every metric returned by the shared evaluator. Prediction equality is reported as a diagnostic, while candidate identity, output artifacts, OOF identity, and exact metric equality are required for PASS.

In [ ]:
with (REPRODUCTION_ROOT / "metrics" / "foundation_promotion_decision.json").open() as handle:
    reproduction_decision = json.load(handle)

actual_candidate = reproduction_decision.get("selected_experiment")
actual_weight = reproduction_decision.get("selected_foundation_weight")
candidate_match = (
    actual_candidate == EXPECTED_CANDIDATE
    and actual_weight == EXPECTED_FOUNDATION_WEIGHT
)

required_artifacts = [
    REPRODUCTION_ROOT / "metrics" / "foundation_preflight.json",
    REPRODUCTION_ROOT / "metrics" / "foundation_final_config.json",
    REPRODUCTION_ROOT / "metrics" / "foundation_experiments.csv",
    REPRODUCTION_ROOT / "metrics" / "foundation_promotion_decision.json",
    REPRODUCTION_ROOT / "metrics" / "foundation_run_manifest.json",
]
for model_name, seeds in {"ctr": (42, 2026), "tabm": (42, 2026), "foundation": (42,)}.items():
    for seed in seeds:
        required_artifacts.extend(
            [
                REPRODUCTION_ROOT / "oof" / f"{model_name}_oof_seed_{seed}.csv",
                REPRODUCTION_ROOT / "oof" / f"{model_name}_test_fold_predictions_seed_{seed}.csv",
                REPRODUCTION_ROOT / "metrics" / f"{model_name}_fold_metrics_seed_{seed}.csv",
            ]
        )
missing_artifacts = [str(path.relative_to(REPRODUCTION_ROOT)) for path in required_artifacts if not path.exists()]

reproduction_oof_path = (
    REPRODUCTION_ROOT / "oof" / f"{actual_candidate}_oof.csv"
    if isinstance(actual_candidate, str)
    else None
)
if reproduction_oof_path is not None and not reproduction_oof_path.exists():
    missing_artifacts.append(str(reproduction_oof_path.relative_to(REPRODUCTION_ROOT)))

reference_metrics = evaluate_probabilities(
    reference_oof[TARGET].to_numpy(dtype=int),
    reference_oof["fraud_probability_raw"].to_numpy(dtype=float),
)
actual_metrics = None
oof_identity_match = False
prediction_exact_match = False
maximum_absolute_prediction_difference = None

if reproduction_oof_path is not None and reproduction_oof_path.exists():
    reproduction_oof = pd.read_csv(reproduction_oof_path)
    required_columns = {ID_COL, TARGET, "fold", "fraud_probability_raw"}
    if not required_columns.issubset(reproduction_oof.columns):
        missing_columns = sorted(required_columns - set(reproduction_oof.columns))
        raise ValueError(f"Reproduction OOF is missing columns: {missing_columns}")
    oof_identity_match = (
        np.array_equal(reproduction_oof[ID_COL].to_numpy(), reference_oof[ID_COL].to_numpy())
        and np.array_equal(reproduction_oof[TARGET].to_numpy(dtype=int), reference_oof[TARGET].to_numpy(dtype=int))
        and np.array_equal(reproduction_oof["fold"].to_numpy(dtype=int), reference_oof["fold"].to_numpy(dtype=int))
    )
    if oof_identity_match:
        reference_predictions = reference_oof["fraud_probability_raw"].to_numpy(dtype=float)
        reproduction_predictions = reproduction_oof["fraud_probability_raw"].to_numpy(dtype=float)
        prediction_exact_match = np.array_equal(reproduction_predictions, reference_predictions)
        maximum_absolute_prediction_difference = float(
            np.max(np.abs(reproduction_predictions - reference_predictions))
        )
        actual_metrics = evaluate_probabilities(
            reproduction_oof[TARGET].to_numpy(dtype=int),
            reproduction_predictions,
        )

metric_rows = []
for metric_name in sorted(reference_metrics):
    expected_value = reference_metrics[metric_name]
    actual_value = actual_metrics.get(metric_name) if actual_metrics is not None else None
    metric_rows.append(
        {
            "metric": metric_name,
            "reference": expected_value,
            "reproduction": actual_value,
            "exact_match": actual_value == expected_value,
        }
    )
metric_comparison = pd.DataFrame(metric_rows)
metrics_match = bool(metric_comparison["exact_match"].all())

failure_reasons = []
if not candidate_match:
    failure_reasons.append("The selected candidate or foundation weight changed.")
if missing_artifacts:
    failure_reasons.append("Required reproduction artifacts are missing.")
if not oof_identity_match:
    failure_reasons.append("The reproduction OOF IDs, labels, or folds differ from the reference.")
if not metrics_match:
    failure_reasons.append("At least one raw OOF metric does not exactly match the reference.")

passed = not failure_reasons
METRICS_OUTPUT = REPRODUCTION_ROOT / "metrics"
metric_comparison.to_csv(METRICS_OUTPUT / "reproduction_metric_comparison.csv", index=False)
verdict = {
    "status": "PASS" if passed else "FAIL",
    "reference_run": REFERENCE_RUN_NAME,
    "reproduction_run": REPRODUCTION_RUN_NAME,
    "candidate": {
        "expected": EXPECTED_CANDIDATE,
        "actual": actual_candidate,
        "expected_foundation_weight": EXPECTED_FOUNDATION_WEIGHT,
        "actual_foundation_weight": actual_weight,
        "passed": candidate_match,
    },
    "artifacts": {"passed": not missing_artifacts, "missing": missing_artifacts},
    "oof_identity_match": oof_identity_match,
    "metrics_exact_match": metrics_match,
    "prediction_diagnostics": {
        "exact_match": prediction_exact_match,
        "maximum_absolute_difference": maximum_absolute_prediction_difference,
    },
    "failure_reasons": failure_reasons,
}
with (METRICS_OUTPUT / "reproduction_verdict.json").open("w") as handle:
    json.dump(verdict, handle, indent=2, sort_keys=True)

display(metric_comparison)
display(pd.DataFrame([verdict["candidate"]]))
print(json.dumps({"status": verdict["status"], "failure_reasons": failure_reasons}, indent=2))

if not passed:
    raise AssertionError("Foundation reproduction failed. See reproduction_verdict.json for details.")